> 在线讲义：[第 5 章 · 一次完整训练：切 batch、每一步做什么、存档、看哪几条曲线](https://wikieden.github.io/beyond-transformer/lessons/05-training-loop.html) · 本书《超越 Transformer》

# 第 5 章 · 一次完整训练：切 batch、每一步做什么、存档、看哪几条曲线

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/05-training-loop.html) · 仓库里 `beyond-transformer/notes/lessons/05-training-loop.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 切 batch | 错开一位 | `windows(STREAM, [1, 6], 4)` | x = [[9, 2, 7, 1], [3, 6, 4, 0]] |
| 第二步 token 账 | B × m × 步数 | `token_budget(64, 512, 10000)` | 327,680,000 |
| 第三步 训练循环 | 五件事 | `train(60)` | w = 2.080、b = 1.079 |
| 第四步 存档 | 全部状态 vs 只存参数 | `resume_demo()` | 一致 vs 对不上 |

checkpoint 要存什么（只存参数、不存优化器状态，续训就和不断的训练对不上）、看哪几条曲线。
a1 的 data_loading、checkpointing、training_together 三个题留给自己写；这里用一个两参数的小模型演示流程。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
STREAM = np.array([5, 9, 2, 7, 1, 8, 3, 6, 4, 0, 2, 9])   # 一长串 token（整份语料拼起来）

**函数 `windows`**：每个起点 s：输入 x = stream[s : s+m]，目标 y = stream[s+1 : s+m+1]（错开一位）。

In [ ]:
def windows(stream, starts, m):
    """每个起点 s：输入 x = stream[s : s+m]，目标 y = stream[s+1 : s+m+1]（错开一位）。"""
    x = np.stack([stream[s : s + m] for s in starts])
    y = np.stack([stream[s + 1 : s + m + 1] for s in starts])
    return x, y

**函数 `token_budget`**：小工具。

In [ ]:
def token_budget(batch, ctx, steps):
    return batch * ctx * steps

**函数 `make_data`**：玩具回归：y = 2x + 1 + 噪声，前 160 个训练、后 40 个验证。

In [ ]:
def make_data(seed=0, n=200):
    """玩具回归：y = 2x + 1 + 噪声，前 160 个训练、后 40 个验证。"""
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1, 1, n)
    y = 2 * x + 1 + 0.1 * rng.normal(size=n)
    return (x[:160], y[:160]), (x[160:], y[160:])

**函数 `train`**：带动量的 SGD 训练 w、b。state = (w, b, 动量 vw, vb, 随机数生成器)；返回新 state 和每步的日志。

In [ ]:
def train(steps, state=None, lr=0.1, beta=0.9, batch=16, seed=1, start=0):
    """带动量的 SGD 训练 w、b。state = (w, b, 动量 vw, vb, 随机数生成器)；返回新 state 和每步的日志。"""
    (xt, yt), (xv, yv) = make_data()
    if state is None:
        state = (0.0, 0.0, 0.0, 0.0, np.random.default_rng(seed))
    w, b, vw, vb, rng = state
    log = []
    for t in range(start, start + steps):
        idx = rng.integers(0, len(xt), batch)
        err = w * xt[idx] + b - yt[idx]
        gw, gb = float(2 * (err * xt[idx]).mean()), float(2 * err.mean())
        vw, vb = beta * vw + gw, beta * vb + gb
        w, b = w - lr * vw, b - lr * vb
        val = float(((w * xv + b - yv) ** 2).mean())
        log.append({"step": t + 1, "train": float((err**2).mean()), "val": val, "gnorm": float(np.hypot(gw, gb))})
    return (w, b, vw, vb, rng), log

**函数 `resume_demo`**：不断地训 60 步，对比：训 30 步存档，完整续训 / 只存参数（动量清零）续训。

In [ ]:
def resume_demo():
    """不断地训 60 步，对比：训 30 步存档，完整续训 / 只存参数（动量清零）续训。"""
    full, _ = train(60)
    half, _ = train(30)
    w, b, vw, vb, rng = half
    rng_state = rng.bit_generator.state
    r1 = np.random.default_rng()
    r1.bit_generator.state = rng_state
    good, _ = train(30, state=(w, b, vw, vb, r1), start=30)
    r2 = np.random.default_rng()
    r2.bit_generator.state = rng_state
    bad, _ = train(30, state=(w, b, 0.0, 0.0, r2), start=30)
    return {"不断训": full[:2], "完整存档续训": good[:2], "只存参数续训": bad[:2]}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 切 batch：输入和目标错开一位")
x, y = windows(STREAM, [1, 6], 4)
print(f"  token 串 {STREAM.tolist()}，起点 [1, 6]、长度 4：\n  x = {x.tolist()}\n  y = {y.tolist()}")
print(f"  长度 12 的串、窗口 4：可选起点 0 到 {len(STREAM) - 4 - 1}，共 {len(STREAM) - 4} 个")
print("\n第二步 token 账")
for b, c, s in ((32, 256, 5000), (64, 512, 10000)):
    print(f"  batch {b} × 上下文 {c} × {s} 步 = {token_budget(b, c, s):,} 个 token")

In [ ]:
print("\n第三步 训练循环与曲线（y = 2x + 1 的小模型）")
(w, b, *_), log = train(60)
for r in log[::15] + [log[-1]]:
    print(f"  第 {r['step']:2d} 步：train {r['train']:.4f}，val {r['val']:.4f}，梯度范数 {r['gnorm']:.4f}")
print(f"  学到 w = {w:.3f}、b = {b:.3f}")

In [ ]:
print("\n第四步 checkpoint 要存什么")
for k, (w, b) in resume_demo().items():
    print(f"  {k}：w = {w:.6f}，b = {b:.6f}")

## 3. 代码里才看得到的

- **随机数状态也是 checkpoint 的一部分**：`resume_demo` 恢复了随机数生成器，续训取到的 batch 才和不断训一样，结果才能逐位一致。
- **这里是两参数的玩具模型**：作业要写的 `get_batch`（返回指定设备上的张量）、`save_checkpoint` / `load_checkpoint` 和完整的训练脚本留给自己写。

## 4. 自己改一改

- 把 `train` 的 lr 改成 0.5，曲线还收敛吗？梯度范数怎么变？
- 把 `resume_demo` 里续训时的随机数生成器换成新的种子（不恢复状态），“完整存档续训”还和不断训一致吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 文档拼成一长串，中间插分隔；随机剪段，输入答案错一位。
> 批乘长乘步数，就是总 token。
> 取批、前向、算 loss，反向、裁剪、更新、清梯度。
> 盯四条线：train、val、梯度范数、学习率；val 一升就是过拟合。
> 存档三件：参数、优化器、步数，少一件就接不上。